# Mean and standard deviation of trajectories

Marianna Georgiou, Department of Psychiatry, Psychotherapy and Psychosomatics, Faculty of Medicine, RWTH Aachen, Aachen, Germany
**Email:** mariannag43@gmail.com
**Publication:**  Latent subtypes of longitudinal postpartum trajectories for identifying healthy women at risk for affective conditions

Summarize seven measures at each available time point as mean ± sample SD, separately by diagnosis and by diagnosis/cluster, for discovery and validation cohorts.

## Inputs

All inputs are in `files/interm_files/`:

- `full_subscales_trajectories_withoutnorm_[1, 2].xlsx` and `full_subscales_trajectories_withoutnorm_[3].xlsx`: raw trajectories by diagnosis.
- `disc_merged_rawdata_withlabels.xlsx` and `val_merged_rawdata_withlabels.xlsx`: raw trajectories with displayed cluster labels 1/2 and diagnosis labels HC/AD/PPD.

The diagnosis tables use original CODE values ND/AS/PPD. Their validation EPDS columns have no `_y` suffix; the clustered-table code expects `_y` for both cohorts. This existing schema assumption is preserved.

## Outputs

- `files/interm_files/Symptom_Trajectories_Compact.xlsx`
- `files/interm_files/Clustered_Symptom_Trajectories_Compact.xlsx`



## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import kruskal
from matplotlib.lines import Line2D
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests
import matplotlib.gridspec as gridspec

%load_ext dotenv
%dotenv
%load_ext autoreload
%autoreload 2

from pathlib import Path

BASE_DIR = Path.cwd()
INPUT_PATH = BASE_DIR / "files" / "inputs"
OUTPUT_FILES = BASE_DIR / "files" / "interm_files"
OUTPUT_PLOTS = BASE_DIR / "files" / "plots"
OUTPUT_TABLES = BASE_DIR / "files" / "tables"
for directory in (OUTPUT_FILES, OUTPUT_PLOTS, OUTPUT_TABLES):
    directory.mkdir(parents=True, exist_ok=True)


## 2. Summaries by diagnosis

Coerce each score to numeric and exclude missing/non-numeric values separately for each summary. Report mean ± sample SD (ddof=1) to two decimal places. Empty groups and MPAS T0 are blank; a single observation gives an undefined SD. Summaries use available values without imputation.

In [ ]:
import pandas as pd
import numpy as np

def format_mean_sd(series):
    """Format available numeric scores as mean ± sample SD."""
    series = pd.to_numeric(series, errors="coerce").dropna()

    if len(series) == 0:
        return ""

    return f"{series.mean():.2f} ± {series.std():.2f}"

def create_compact_table(coh, cohort_name):
    """Summarize raw scores by diagnosis."""

    input_raw = OUTPUT_FILES / f'full_subscales_trajectories_withoutnorm_{coh}.xlsx'

    df = pd.read_excel(input_raw)

    # Use the EPDS column names expected in this input workbook.
    if coh == [1, 2]:
        epds_columns = [
            "EPDStotal_t0_y",
            "EPDStotal_t1_y",
            "EPDStotal_t2_y",
            "EPDStotal_t3_y",
            "EPDStotal_t4_y"
        ]

    elif coh == [3]:
        epds_columns = [
            "EPDStotal_t0",
            "EPDStotal_t1",
            "EPDStotal_t2",
            "EPDStotal_t3",
            "EPDStotal_t4"
        ]

    measures = {
        "EPDS": epds_columns,

        "Anhedonia": [
            "Anhedonie_t0",
            "Anhedonie_t1",
            "Anhedonie_t2",
            "Anhedonie_t3",
            "Anhedonie_t4"
        ],

        "Anxiety": [
            "Anxiety_t0",
            "Anxiety_t1",
            "Anxiety_t2",
            "Anxiety_t3",
            "Anxiety_t4"
        ],

        "Depression": [
            "Depression_t0",
            "Depression_t1",
            "Depression_t2",
            "Depression_t3",
            "Depression_t4"
        ],

        "Mood": [
            "Mood T0",
            "Mood T1",
            "Mood T2",
            "Mood T3",
            "Mood T4"
        ],

        "Stress": [
            "Stress T0",
            "Stress T1",
            "Stress T2",
            "Stress T3",
            "Stress T4"
        ],

        "MPAS": [
            None,
            "MPAS T1",
            "MPAS T2",
            "MPAS T3",
            "MPAS T4"
        ]
    }

    rows = []

    for measure, columns in measures.items():

        for i, column in enumerate(columns):

            row = {
                "Measure": f"{measure} T{i}"
            }

            for code in ["ND", "AS", "PPD"]:

                code_df = df[df["CODE"] == code]

                if column is None:
                    row[code] = ""
                else:
                    row[code] = format_mean_sd(
                        code_df[column]
                    )

            rows.append(row)

    table = pd.DataFrame(rows)

    return table

discovery_table = create_compact_table(
    [1, 2],
    "Discovery"
)

validation_table = create_compact_table(
    [3],
    "Validation"
)

output_path = OUTPUT_FILES / "Symptom_Trajectories_Compact.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

    discovery_table.to_excel(
        writer,
        sheet_name="Discovery",
        index=False
    )

    validation_table.to_excel(
        writer,
        sheet_name="Validation",
        index=False
    )

discovery_table


### Inspect a clustered input

Load the validation table for inspection before generating both cohorts’ clustered summaries.

In [ ]:
coh = [3]
if coh == [3]:
    input_raw = OUTPUT_FILES / "val_merged_rawdata_withlabels.xlsx"
elif coh == [1, 2]:
    input_raw = OUTPUT_FILES / "disc_merged_rawdata_withlabels.xlsx"

df = pd.read_excel(input_raw)


In [ ]:
df.columns


## 3. Summaries by diagnosis and cluster

Summarize HC C1, HC C2, AD C1, AD C2, and PPD C2; PPD C1 is not included in the fixed group list. This cell redefines `create_compact_table`, so run the notebook from the top when regenerating both workbooks. The `cohort_name` argument is retained but unused.

In [ ]:
def create_compact_table(coh, cohort_name):
    """Summarize raw scores for the five specified diagnosis/cluster groups."""
    if coh == [3]:
            input_raw = OUTPUT_FILES / "val_merged_rawdata_withlabels.xlsx"
    elif coh == [1, 2]:
            input_raw = OUTPUT_FILES / "disc_merged_rawdata_withlabels.xlsx"

    df = pd.read_excel(input_raw)
    df['Group'] = df.apply(lambda row: f"{row['CODE']} (C{row['IDX']})", axis=1)

    # Use the EPDS column names expected in this input workbook.
    if coh == [1, 2]:
        epds_columns = [
            "EPDStotal_t0_y",
            "EPDStotal_t1_y",
            "EPDStotal_t2_y",
            "EPDStotal_t3_y",
            "EPDStotal_t4_y"
        ]

    elif coh == [3]:
        epds_columns = [
            "EPDStotal_t0_y",
            "EPDStotal_t1_y",
            "EPDStotal_t2_y",
            "EPDStotal_t3_y",
            "EPDStotal_t4_y"
        ]

    measures = {
        "EPDS": epds_columns,

        "Anhedonia": [
            "Anhedonie_t0",
            "Anhedonie_t1",
            "Anhedonie_t2",
            "Anhedonie_t3",
            "Anhedonie_t4"
        ],

        "Anxiety": [
            "Anxiety_t0",
            "Anxiety_t1",
            "Anxiety_t2",
            "Anxiety_t3",
            "Anxiety_t4"
        ],

        "Depression": [
            "Depression_t0",
            "Depression_t1",
            "Depression_t2",
            "Depression_t3",
            "Depression_t4"
        ],

        "Mood": [
            "Mood T0",
            "Mood T1",
            "Mood T2",
            "Mood T3",
            "Mood T4"
        ],

        "Stress": [
            "Stress T0",
            "Stress T1",
            "Stress T2",
            "Stress T3",
            "Stress T4"
        ],

        "MPAS": [
            None,
            "MPAS T1",
            "MPAS T2",
            "MPAS T3",
            "MPAS T4"
        ]
    }

    rows = []

    for measure, columns in measures.items():

        for i, column in enumerate(columns):

            row = {
                "Measure": f"{measure} T{i}"
            }

            for group in ["HC (C1)", "HC (C2)", "AD (C1)", "AD (C2)", "PPD (C2)"]:

                group_df = df[df["Group"] == group]

                if column is None:
                    row[group] = ""
                else:
                    row[group] = format_mean_sd(
                        group_df[column]
                    )

            rows.append(row)

    table = pd.DataFrame(rows)

    return table

discovery_table = create_compact_table(
    [1, 2],
    "Discovery"
)

validation_table = create_compact_table(
    [3],
    "Validation"
)

output_path = OUTPUT_FILES / "Clustered_Symptom_Trajectories_Compact.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

    discovery_table.to_excel(
        writer,
        sheet_name="Discovery",
        index=False
    )

    validation_table.to_excel(
        writer,
        sheet_name="Validation",
        index=False
    )

discovery_table


In [ ]:
OUTPUT_FILES
